# Paper QA：向量化与检索演示

这个 notebook 演示第一阶段（A. retrieve）里"文本变成向量、再用向量找 chunk"的全过程。
所有逻辑在 `scripts/retrieval.py`，这里只调用它，并把每一步的中间结果摆出来。

内容：
1. 一个 chunk 在送进模型之前长什么样（`embed_text`）
2. 向量存在哪里：`embeddings.npy`、`chunk_ids.json`、`meta.json`
3. 存下来的向量和重新编码的结果是否一致
4. 手工做一遍检索（query 向量 × 矩阵），并和 `Retriever.retrieve` 对照
5. 几个示例查询
6. 限定论文范围、`lookup_citation`
7. 在评测集 dev 上看检索的真实表现，以及一个失败例子

运行前：选择项目 `.venv` 的 Python 内核。索引已经建好，不需要重建（重建命令：`python -m scripts.retrieval build`，约 4 分钟）。
第 4 步起会加载 bge-m3 模型（有 GPU 时用 GPU，约十几秒）。

In [1]:
import json
import sys
from pathlib import Path

ROOT = Path.cwd()
while not (ROOT / 'scripts').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import numpy as np
from scripts import retrieval as rt

print('项目根目录:', ROOT)

项目根目录: c:\Users\boyue\Desktop\LLM projects\Paper_QA_RAG


## 1. 一个 chunk 送进模型之前长什么样

chunk 文件是冻结的分块结果（5755 个）。编码时不是只用正文，而是把"论文标题 | 章节路径"拼在正文前面，
这样模型能看到这段话属于哪篇论文的哪一节。拼接只发生在编码那一刻，存储的 chunk 原文没有被改动。

In [2]:
run_id, chunks_path = rt.current_run()
chunks = rt.read_chunks(chunks_path)
by_id = {c['chunk_id']: c for c in chunks}
print('分块 run id:', run_id)
print('chunk 总数 :', len(chunks))

chunk = chunks[100]
print()
print('一个 chunk 的字段:', list(chunk))
print()
for key in ('chunk_id', 'paper_id', 'paper_title', 'section_path', 'section_title','text'):
    print(f'{key:14}', chunk[key])

分块 run id: section_recursive_512_64_e14374fa6d
chunk 总数 : 5755

一个 chunk 的字段: ['chunk_id', 'paper_id', 'arxiv_id', 'paper_title', 'citation', 'version', 'source_format', 'source_url', 'source_path', 'source_sha256', 'section_id', 'section_title', 'section_path', 'section_kind', 'merged_section_titles', 'section_chunk_index', 'char_start', 'char_end', 'offset_scope', 'text', 'token_count', 'tokenizer', 'page_numbers', 'content_types', 'source_spans', 'parser_quality_flags', 'chunker_version', 'config_id', 'previous_chunk_id', 'next_chunk_id']

chunk_id       chunk_535ef1e08ea167d16322cb7e
paper_id       arxiv_1701.06538
paper_title    Outrageously Large Neural Networks: The Sparsely-Gated Mixture-of-Experts Layer
section_path   ['5 Experiments', '5.1 1 Billion Word Language Modeling Benchmark']
section_title  5.1 1 Billion Word Language Modeling Benchmark
text           we include the importance-sampling-based training of the softmax layer, and we count a multiply-and-add as two separat

In [3]:
text_for_model = rt.embed_text(chunk)
print(text_for_model[:700])
print('...')
print()
print('正文字符数:', len(chunk['text']), '| 送进模型的字符数:', len(text_for_model))

Outrageously Large Neural Networks: The Sparsely-Gated Mixture-of-Experts Layer | 5 Experiments > 5.1 1 Billion Word Language Modeling Benchmark
we include the importance-sampling-based training of the softmax layer, and we count a multiply-and-add as two separate operations. For all of our MoE models, the floating point operations involved in the experts represent between 37% and 46% of the total.

For our baseline models wtih no MoE, observed computational efficiency ranged from 1.07-1.29 TFLOPS/GPU. For our low-computation MoE models, computation efficiency ranged from 0.74-0.90 TFLOPS/GPU, except for the 4-expert model which did not make full use of the available parallelism. Our highest
...

正文字符数: 808 | 送进模型的字符数: 953


## 2. 向量存在哪里

索引目录名就是分块 run id，所以换了分块就不会误用旧索引。三个文件：

- `embeddings.npy`：N×1024 的矩阵，**第 i 行就是第 i 个 chunk 的向量**，矩阵里没有 id
- `chunk_ids.json`：长度 N 的列表，第 i 项是第 i 行对应的 chunk_id（靠下标对应）
- `meta.json`：模型、维度、是否归一化、拼接格式、chunk 文件哈希

In [5]:
index_dir = rt.INDEX_ROOT / run_id
for path in sorted(index_dir.iterdir()):
    print(f'{path.name:18}{path.stat().st_size / 1e6:8.2f} MB')

print()
print(json.dumps(json.loads((index_dir / 'meta.json').read_text(encoding='utf-8')), indent=2, ensure_ascii=False))

chunk_ids.json        0.20 MB
embeddings.npy       23.57 MB
meta.json             0.00 MB

{
  "model": "BAAI/bge-m3",
  "run_id": "section_recursive_512_64_e14374fa6d",
  "embed_format": "v1: \"{paper_title} | {section_path joined by >}\\n{text}\"",
  "normalized": true,
  "count": 5755,
  "dim": 1024,
  "max_embed_tokens": 594,
  "device": "cuda:0",
  "seconds": 236.0,
  "chunks_sha256": "b7f17a680500912c26caa8eeb4bf49a159c7c72f960ed0b17ebf3583984bd995",
  "partial": false
}


In [ ]:
vectors = np.load(index_dir / 'embeddings.npy')
chunk_ids = json.loads((index_dir / 'chunk_ids.json').read_text(encoding='utf-8'))

In [7]:
vectors

array([[-0.03062439, -0.02874756, -0.02580261, ..., -0.02177429,
        -0.01663208,  0.03134155],
       [-0.03781128, -0.00535202, -0.02874756, ..., -0.01078033,
         0.02078247,  0.03335571],
       [-0.04290771, -0.00539398, -0.0287323 , ..., -0.01126099,
         0.01979065,  0.01190186],
       ...,
       [-0.02352905, -0.04699707, -0.00661087, ..., -0.02630615,
         0.03442383,  0.01757812],
       [-0.08483887, -0.02198792, -0.01100159, ..., -0.02011108,
         0.05276489, -0.02828979],
       [-0.07025146, -0.04226685, -0.00568771, ..., -0.02597046,
         0.02656555,  0.00319862]], shape=(5755, 1024), dtype=float32)

In [8]:
chunk_ids

['chunk_cf4c20caebebb66002b1f5d7',
 'chunk_fc3247f1eed4e2e4c0cda22c',
 'chunk_94b72d4d7d33861782f97938',
 'chunk_518af47555553d2379a986ef',
 'chunk_11046708878e37816b936edc',
 'chunk_1dcdc6c6b3eca1b877b8f9b9',
 'chunk_6da1107f5769714ba97814eb',
 'chunk_6320d4a538caf0fb5e161b18',
 'chunk_7c2069afc05e6588b99b3beb',
 'chunk_ee43d91f0f2fb32b63039275',
 'chunk_2177640e9e98a5f9768b3099',
 'chunk_97781968c99ec556590e27e9',
 'chunk_f461c4e0437289b26797e465',
 'chunk_13e616f5296422b8bd0b6e29',
 'chunk_b295662f07d04927cb9dabac',
 'chunk_9fb82d61d006a45672cd0188',
 'chunk_22666d7957e079438572ef40',
 'chunk_60711dfa200850a1fb85b088',
 'chunk_1b211b45b2179c797efca30f',
 'chunk_d93ba607aeb90211b4627e2f',
 'chunk_28064ba2a1fccc1b9e1bd57e',
 'chunk_95c1a0ec87d25b29fa27ee7f',
 'chunk_f2d67241c58364b8a9134172',
 'chunk_2f88206a66f9fcac95da9235',
 'chunk_1cd081671db801a8b34c7108',
 'chunk_270bbca0657b7d23329c9e43',
 'chunk_aa6232f09768c1ba378b79e9',
 'chunk_d4098f46850b0843e1cdff57',
 'chunk_448fac111cd5

In [9]:
print('矩阵形状:', vectors.shape, vectors.dtype)
print('chunk_ids 长度:', len(chunk_ids), '| 前 3 个:', chunk_ids[:3])

norms = np.linalg.norm(vectors, axis=1)
print(f'向量模长: 最小 {norms.min():.6f}, 最大 {norms.max():.6f}  (约等于 1，说明已归一化；微小偏差来自 fp16)')
print()
print(f'第1000行 -> chunk_id {chunk_ids[999]} -> {by_id[chunk_ids[999]]["paper_title"]} / {by_id[chunk_ids[999]]["section_title"]}')
print('向量:', np.round(vectors[999], 4))

矩阵形状: (5755, 1024) float32
chunk_ids 长度: 5755 | 前 3 个: ['chunk_cf4c20caebebb66002b1f5d7', 'chunk_fc3247f1eed4e2e4c0cda22c', 'chunk_94b72d4d7d33861782f97938']
向量模长: 最小 0.999756, 最大 1.000486  (约等于 1，说明已归一化；微小偏差来自 fp16)

第1000行 -> chunk_id chunk_ac1d07bffb1e5f144cc4ab03 -> Sophia: A Scalable Stochastic Second-order Optimizer for Language Model Pre-training / 2 Method
向量: [-0.0699 -0.0209 -0.0182 ... -0.0118 -0.0281 -0.0047]


## 3. 存下来的向量和重新编码一致吗

随机抽 10 个 chunk，用同一个模型重新编码，和存储的向量算点积。应该非常接近 1
（GPU 上用 fp16，所以不会是精确的 1.0000）。

In [10]:
retriever = rt.Retriever()
model = retriever._model or rt.load_model()
retriever._model = model
print('模型所在设备:', model.device)

rng = np.random.default_rng(0)
rows = rng.choice(len(chunks), 10, replace=False)
fresh = model.encode([rt.embed_text(by_id[chunk_ids[r]]) for r in rows], normalize_embeddings=True, convert_to_numpy=True)
same = (fresh * vectors[rows]).sum(axis=1)
print('重新编码与存储向量的相似度:', np.round(same, 4))

c:\Users\boyue\Desktop\LLM projects\Paper_QA_RAG\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 391/391 [00:00<00:00, 17452.46it/s]


模型所在设备: cuda:0
重新编码与存储向量的相似度: [0.9997 1.     1.0003 1.0004 0.9997 1.0001 1.0008 1.0003 1.0004 1.001 ]


## 4. 手工做一遍检索

步骤：
1. query 用同一个模型编码成 1024 维向量（同样归一化）
2. 向量矩阵 × query 向量，得到 5755 个相似度
3. 取最大的 k 个，行号换成 chunk_id，再取出正文

In [12]:
query = 'Why does the Transformer use multi-head attention?'

q = retriever.encode(query)
scores = vectors @ q
print('相似度个数:', scores.shape, f'| 最高 {scores.max():.3f} | 中位数 {np.median(scores):.3f} | 最低 {scores.min():.3f}')

top_rows = np.argsort(-scores)[:5]
for rank, r in enumerate(top_rows, 1):
    c = by_id[chunk_ids[r]]
    print(f'{rank}. {scores[r]:.3f}  {c["paper_title"]} | {c["section_title"]}')

相似度个数: (5755,) | 最高 0.722 | 中位数 0.400 | 最低 0.235
1. 0.722  Attention Is All You Need | 3.2.3 Applications of Attention in our Model
2. 0.694  Attention Is All You Need | 2 Background
3. 0.692  Formal Algorithms for Transformers | 5 Architectural Components
4. 0.690  Attention Is All You Need | 7 Conclusion
5. 0.674  Transformers are SSMs: Generalized Models and Efficient Algorithms Through Structured State Space Duality | 9.2 Language Modeling


In [13]:
manual = [chunk_ids[r] for r in top_rows]
official = [h['chunk_id'] for h in retriever.retrieve(query, k=5)]
print('手工结果与 Retriever.retrieve 一致:', manual == official)

手工结果与 Retriever.retrieve 一致: True


## 5. 几个示例查询

`retrieve` 返回的每一项含：`chunk_id`、`score`、`paper_id`、`paper_title`、`citation`、`section_title`、`text`。
下面换几种问法，只看分数、论文、章节和正文开头。

In [14]:
def show(query, k=5, **kwargs):
    print('Q:', query)
    for rank, h in enumerate(retriever.retrieve(query, k, **kwargs), 1):
        print(f'  {rank}. {h["score"]:.3f} {h["paper_title"][:40]:40} | {h["section_title"][:28]:28} | {h["text"][:70]!r}')
    print()

show('What learning rate schedule does the original Transformer use?')
show('How does layer normalization differ from batch normalization?')
show('Which paper introduced low-rank adaptation for fine-tuning large language models?')

Q: What learning rate schedule does the original Transformer use?
  1. 0.619 Attention Is All You Need                | 6.1 Machine Translation      | 'Table 2: The Transformer achieves better BLEU scores than previous sta'
  2. 0.613 Attention Is All You Need                | Abstract                     | 'The dominant sequence transduction models are based on complex recurre'
  3. 0.612 Transformers are RNNs: Fast Autoregressi | Appendix B Training Evolutio | 'In figure 5 we present the training evolution of all transformer model'
  4. 0.612 Attention Is All You Need                | 6.1 Machine Translation      | '10^{21}$\nTransformer (base model) | 27.3 | 38.1 |  | $3.3\\cdot 10^{18}'
  5. 0.609 Attention Is All You Need                | 7 Conclusion                 | 'In this work, we presented the Transformer, the first sequence transdu'

Q: How does layer normalization differ from batch normalization?
  1. 0.715 Layer Normalization                      | Abstract              

最后一个是问"哪篇论文"，检索结果的论文标题可以直接回答；前两个则要看具体 chunk 里有没有所问的事实。

**无关问题**也会返回 top-k，只是分数更低。相似度没有一个通用的阈值能区分"有答案"和"没答案"，
所以后面是否拒答靠起草和验证，而不是靠分数。

In [15]:
show('How many tokens was Mixtral 8x7B pretrained on?')
show('What is the best recipe for chocolate cake?')

Q: How many tokens was Mixtral 8x7B pretrained on?
  1. 0.637 Mixtral of Experts                       | 6 Conclusion                 | 'In this paper, we introduced Mixtral 8x7B, the first mixture-of-expert'
  2. 0.594 Mixtral of Experts                       | Abstract                     | 'We introduce Mixtral 8x7B, a Sparse Mixture of Experts (SMoE) language'
  3. 0.592 Mixtral of Experts                       | 1 Introduction               | 'In this paper, we present Mixtral 8x7B, a sparse mixture of experts mo'
  4. 0.537 Mixtral of Experts                       | 3 Results                    | 'vs Llama 2 (7B/13B/70B). Mixtral largely outperforms Llama 2 70B on al'
  5. 0.533 Mixtral of Experts                       | 1 Introduction               | 'that of GPT-3.5 Turbo, Claude-2.1, Gemini Pro, and Llama 2 70B – chat '

Q: What is the best recipe for chocolate cake?
  1. 0.506 Large Language Models are Zero-Shot Reas | A.6 Answer Cleansing         | 'boxes of caramel candy. I

## 6. 限定论文范围、查引用信息

`retrieve(..., paper_ids=[...])` 只在指定论文里找。`lookup_citation(chunk_id)` 是第二个工具，
按 chunk_id 取回引用信息，以及相邻 chunk 的 id；id 不存在时返回 `None`，验证器靠这一点发现编造的引用。

In [11]:
papers = sorted({c['paper_id'] for c in chunks})
print('论文数:', len(papers), '| 例如:', papers[:3])

target = by_id[chunk_ids[top_rows[0]]]['paper_id']
print('只在', target, '里检索:')
show(query, k=3, paper_ids=[target])

论文数: 78 | 例如: ['arxiv_1503.02531', 'arxiv_1508.07909', 'arxiv_1607.06450']
只在 arxiv_1706.03762 里检索:
Q: Why does the Transformer use multi-head attention?


  1. 0.722 Attention Is All You Need                | 3.2.3 Applications of Attent | 'The Transformer uses multi-head attention in three different ways:\n\n• '
  2. 0.694 Attention Is All You Need                | 2 Background                 | 'The goal of reducing sequential computation also forms the foundation '
  3. 0.690 Attention Is All You Need                | 7 Conclusion                 | 'In this work, we presented the Transformer, the first sequence transdu'



In [12]:
info = retriever.lookup_citation(chunk_ids[top_rows[0]])
print({k: (v if k != 'text' else v[:80] + '...') for k, v in info.items()})
print()
print('不存在的 id ->', retriever.lookup_citation('chunk_does_not_exist'))

{'chunk_id': 'chunk_ee43d91f0f2fb32b63039275', 'paper_id': 'arxiv_1706.03762', 'arxiv_id': '1706.03762', 'paper_title': 'Attention Is All You Need', 'citation': 'Vaswani et al. (2017)', 'version': 'v7', 'section_path': ['3 Model Architecture', '3.2 Attention', '3.2.3 Applications of Attention in our Model'], 'source_url': 'https://arxiv.org/html/1706.03762v7', 'element_ids': ['S3.SS2.SSS3.p1.1', 'S3.I1.i1', 'S3.I1.i2', 'S3.I1.i3'], 'previous_chunk_id': None, 'next_chunk_id': None, 'text': 'The Transformer uses multi-head attention in three different ways:\n\n• In "encode...'}

不存在的 id -> None


## 7. 在评测集 dev 上看检索的真实表现

评测题的金标是原文里的逐字引文，已经对齐到 chunk id（`matched_chunk_ids`）。
"要点可达率" = 每个要点只要有一条证据所在的 chunk 被检索到就算命中，再对要点取平均。
只用检索，不调用大模型。

In [13]:
from scripts.evaluate import default_questions, gold_chunks, key_point_reach, load_questions

questions = [q for q in load_questions(default_questions(), 'dev') if q['type'] != 'unanswerable']
print('dev 可回答题:', len(questions))

reach = {}
ranks = {}
for q in questions:
    got = [h['chunk_id'] for h in retriever.retrieve(q['question'], 20)]
    gold = gold_chunks(q)
    ranks[q['question_id']] = next((i for i, c in enumerate(got, 1) if c in gold), None)
    for k in (5, 8, 20):
        reach.setdefault((q['type'], k), []).append(key_point_reach(q, got[:k]))

print()
print(f'{"题型":14}{"@5":>8}{"@8":>8}{"@20":>8}')
for t in ('single_fact', 'multi_paper', 'comparison'):
    print(f'{t:14}' + ''.join(f'{np.mean(reach[(t, k)]):8.2f}' for k in (5, 8, 20)))

dev 可回答题: 24



题型                  @5      @8     @20
single_fact       0.81    0.81    1.00
multi_paper       0.50    0.62    0.67
comparison        0.58    0.72    0.85


单事实题检索基本够用，多论文和对比题在 top-8 里覆盖不全。这正是后面把问题拆成子问题、
再补检索的动机。

In [14]:
worst = min(questions, key=lambda q: (key_point_reach(q, [h["chunk_id"] for h in retriever.retrieve(q["question"], 8)]) or 0, q["question_id"]))
print(worst['question_id'], worst['type'])
print('问题:', worst['question'])
print('金标证据:')
for e in worst['evidence'][:3]:
    print('  -', e['paper_id'], '|', e['section_title'], '|', e['quote'][:90])

hits = retriever.retrieve(worst['question'], 20)
gold = gold_chunks(worst)
print()
print('top-8 检索到的:')
for rank, h in enumerate(hits[:8], 1):
    print(f'  {rank}. {h["score"]:.3f} {"[金标]" if h["chunk_id"] in gold else "      "} {h["paper_title"][:38]:38} | {h["section_title"][:28]}')
first = next((i for i, h in enumerate(hits, 1) if h['chunk_id'] in gold), None)
print('第一个金标 chunk 的排名 (top-20 内):', first)

q001 single_fact
问题: According to Kaplan et al., if model size is increased 8x, roughly how much must the dataset grow to avoid an overfitting penalty, and why?
金标证据:
  - arxiv_2001.08361 | 1.1 Summary | meaning that every time we increase the model size 8x, we only need to increase the data b
  - arxiv_2001.08361 | 1.2 Summary of Scaling Laws | as we increase the model size, we should increase the dataset size sublinearly

top-8 检索到的:
  1. 0.604        Training Compute-Optimal Large Languag | D.4 Small-scale comparison t
  2. 0.600        Scaling Laws for Neural Language Model | 4.1 Proposed $L(N,D)$ Equati
  3. 0.593        Training Compute-Optimal Large Languag | 1 Introduction
  4. 0.591        Scaling Laws for Neural Language Model | 4.2 Results
  5. 0.591        Scaling Laws for Neural Language Model | 6.3 Contradictions and a Con
  6. 0.586        Scaling Laws for Neural Language Model | 5.3 Lower Bound on Early Sto
  7. 0.586        Scaling Laws for Neural Language Model | 4.2 

## 小结

- 向量存成 `embeddings.npy`（按行号对应）+ `chunk_ids.json`（行号 → chunk_id）
- 编码用 bge-m3，输入是"标题 | 章节路径 + 正文"，向量已归一化，余弦相似度就是点积
- 检索 = query 编码 → 矩阵乘法 → 取 top-k → 按 chunk_id 取正文
- 已知限制：只用稠密向量，对换了说法的问题、对比题、多论文题召回不全；相似度分数不能当拒答依据